# **google/electra-base-discriminator**




### Required Libraries

- **transformers** – Provides pretrained models and tokenizers for NLP tasks.
- **datasets** – Used to load and preprocess datasets efficiently.
- **seqeval** – Calculates NER evaluation metrics such as Precision, Recall, and F1-score.
- **scikit-learn** – Provides utilities for data splitting and machine learning tasks.
- **torch** – PyTorch framework used for building and training deep learning models.
- **accelerate** – Enables faster and more efficient model training.

In [1]:
!pip install -q transformers datasets seqeval scikit-learn torch accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


### Library Description

- **json** – Reads and writes JSON files.
- **random** – Generates random numbers and helps shuffle data.
- **os** – Interacts with the operating system and file paths.
- **Counter** – Counts the frequency of elements in a collection.
- **NumPy** – Performs numerical computations on arrays.
- **PyTorch** – Provides tensors and deep learning functionalities.
- **Dataset** – Creates custom datasets for model training.
- **AutoTokenizer** – Converts text into tokens that the model can understand.
- **AutoModelForTokenClassification** – Loads a pretrained model for Named Entity Recognition (NER).
- **TrainingArguments** – Defines training parameters such as batch size and learning rate.
- **Trainer** – Handles the training and evaluation process.
- **EarlyStoppingCallback** – Stops training when validation performance stops improving.
- **classification_report** – Generates a detailed performance report.
- **precision_score** – Measures the accuracy of positive predictions.
- **recall_score** – Measures how many actual entities are correctly identified.
- **f1_score** – Computes the harmonic mean of precision and recall.

In [2]:
import json
import random
import os
from collections import Counter

import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

print("Imports OK")
print(f"Torch version : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")


Imports OK
Torch version : 2.11.0+cu128
CUDA available: True
GPU           : Tesla T4


## Model Configuration

This section defines the training configuration and hyperparameters used throughout the experiment. It specifies the pretrained model, dataset path, data split ratios, training parameters, and entity labels.

### Parameter Description

- **BASE_MODEL** – Pretrained transformer model used as the backbone for NER.
- **DATA_PATH** – Location of the dataset file.
- **OUTPUT_DIR** – Directory where the trained model and checkpoints are saved.
- **MAX_LENGTH** – Maximum number of tokens processed per sentence.
- **SEED** – Random seed for reproducibility.
- **TEST_SPLIT** – Fraction of data reserved for final testing.
- **VAL_SPLIT** – Fraction of training data used for validation.
- **LEARNING_RATE** – Controls how quickly the model updates its weights.
- **BATCH_SIZE** – Number of samples processed in one training step.
- **GRAD_ACCUM_STEPS** – Simulates a larger batch size by accumulating gradients.
- **NUM_EPOCHS** – Number of times the model sees the entire training dataset.
- **WARMUP_RATIO** – Gradually increases the learning rate during the initial training phase.
- **WEIGHT_DECAY** – Regularization term used to reduce overfitting.
- **EARLY_STOP_PAT** – Stops training if validation performance does not improve.

### Entity Settings

- **ENTITY_TAGS** – List of entity types used in Named Entity Recognition.
- **RARE_TAGS** – Entity categories with fewer examples in the dataset.
- **RARE_WEIGHT** – Additional weight assigned to rare entity types to improve their learning.

### Batch Size and Gradient Accumulation

Batch size determines how many samples are processed in a single step. Gradient accumulation combines gradients from multiple mini-batches before updating the model weights, allowing a larger effective batch size while reducing GPU memory requirements.

Effective Batch Size = Batch Size × Gradient Accumulation Steps

In [3]:
# ============================================================
# CONFIG — edit DATA_PATH to your actual file
# ============================================================
BASE_MODEL        = "google/electra-base-discriminator"   # swap to "tner/deberta-v3-large-ontonotes5" if you want
DATA_PATH         = "final_dataset.json"   # <-- change this
OUTPUT_DIR        = "./ner_output"

MAX_LENGTH        = 256     # was 128 — see Cell 10 for why this matters
SEED              = 42
TEST_SPLIT        = 0.10    # held out, never touched until Cell 21
VAL_SPLIT         = 0.10    # taken from the remaining pool after test is removed

LEARNING_RATE     = 2e-5
BATCH_SIZE        = 8       # small per-step batch to avoid OOM at MAX_LENGTH=256
GRAD_ACCUM_STEPS  = 2      # effective batch size = BATCH_SIZE * GRAD_ACCUM_STEPS
NUM_EPOCHS        = 8
WARMUP_RATIO      = 0.06
WEIGHT_DECAY      = 0.01
EARLY_STOP_PAT    = 2

ENTITY_TAGS = [
    "PER", "ORG", "LOC", "FAC", "DATE", "TIME", "EVT",
    "PROD", "MONEY", "PCT", "LAW", "NORP", "LANG",
    "WOA", "QTY", "ORD", "CARD",
]
RARE_TAGS   = {"WOA", "EVT", "LANG", "QTY", "TIME", "ORD", "CARD", "FAC"}
RARE_WEIGHT = 3.0

print(f"Backbone      : {BASE_MODEL}")
print(f"MAX_LENGTH    : {MAX_LENGTH}  (raised from 128)")
print(f"Effective batch size : {BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"Entity tags ({len(ENTITY_TAGS)} total): {ENTITY_TAGS}")


Backbone      : google/electra-base-discriminator
MAX_LENGTH    : 256  (raised from 128)
Effective batch size : 8 x 2 = 16
Entity tags (17 total): ['PER', 'ORG', 'LOC', 'FAC', 'DATE', 'TIME', 'EVT', 'PROD', 'MONEY', 'PCT', 'LAW', 'NORP', 'LANG', 'WOA', 'QTY', 'ORD', 'CARD']


## Random Seed and Label Mapping

This section ensures reproducible experiments and creates the label mappings required for Named Entity Recognition (NER).

### Random Seed

A fixed random seed is used to ensure consistent results across multiple runs. It initializes the random generators of Python, NumPy, and PyTorch, making the training process more reproducible and allowing fair comparison between experiments.

### Label Mapping

Since neural networks can only process numerical values, entity labels must be converted into integer IDs. Two dictionaries are created:

- **label2id**: Maps each label to a unique integer for training.
- **id2label**: Converts predicted IDs back into human-readable labels.

The model follows the **BIO tagging scheme**, where:

- **B-** indicates the beginning of an entity.
- **I-** indicates the continuation of the same entity.
- **O** represents tokens that do not belong to any entity.

These mappings define the complete output space of the model and determine the size of the classifier head.

In [4]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

def build_label_maps():
    labels = ["O"]
    for tag in ENTITY_TAGS:
        labels.append(f"B-{tag}")
        labels.append(f"I-{tag}")
    label2id = {l: i for i, l in enumerate(labels)}
    id2label = {i: l for l, i in label2id.items()}
    return label2id, id2label

label2id, id2label = build_label_maps()

print(f"Total label classes: {len(label2id)}  (1 'O' + {len(ENTITY_TAGS)} tags x B/I)")
print("\nFull label set fed to the model's classifier head:")
for i, l in id2label.items():
    print(f"  {i:2d} -> {l}")


Total label classes: 35  (1 'O' + 17 tags x B/I)

Full label set fed to the model's classifier head:
   0 -> O
   1 -> B-PER
   2 -> I-PER
   3 -> B-ORG
   4 -> I-ORG
   5 -> B-LOC
   6 -> I-LOC
   7 -> B-FAC
   8 -> I-FAC
   9 -> B-DATE
  10 -> I-DATE
  11 -> B-TIME
  12 -> I-TIME
  13 -> B-EVT
  14 -> I-EVT
  15 -> B-PROD
  16 -> I-PROD
  17 -> B-MONEY
  18 -> I-MONEY
  19 -> B-PCT
  20 -> I-PCT
  21 -> B-LAW
  22 -> I-LAW
  23 -> B-NORP
  24 -> I-NORP
  25 -> B-LANG
  26 -> I-LANG
  27 -> B-WOA
  28 -> I-WOA
  29 -> B-QTY
  30 -> I-QTY
  31 -> B-ORD
  32 -> I-ORD
  33 -> B-CARD
  34 -> I-CARD


## Dataset Loading

This section checks whether the specified dataset file exists. If the file is not found and the code is running in Google Colab, an upload dialog is opened to allow manual file upload. The file path and size are then displayed to verify that the dataset has been loaded successfully.

In [5]:
# Loads DATA_PATH. On Colab, opens an upload dialog if the file isn't found.
if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        print(f"'{DATA_PATH}' not found locally. Opening upload dialog...")
        uploaded = files.upload()
        DATA_PATH = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(
            f"'{DATA_PATH}' not found and google.colab is unavailable. "
            f"Upload the file manually and set DATA_PATH to its path."
        )

print(f"Using data file: {DATA_PATH}")
print(f"File size: {os.path.getsize(DATA_PATH) / 1024:.1f} KB")


Using data file: final_dataset.json
File size: 49837.7 KB


## Parsing Label Studio Annotations

This section converts the raw Label Studio JSON export into a structured format suitable for Named Entity Recognition (NER) training.

The `parse_label_studio()` function reads the dataset file and extracts the text along with its entity annotations. Each entity is represented as a tuple containing:

- **start position** – character index where the entity begins.
- **end position** – character index where the entity ends.
- **label** – entity category (PER, ORG, LOC, etc.).

Duplicate and empty texts are skipped to prevent redundant training samples and improve dataset quality.

Only labels defined in `ENTITY_TAGS` are retained, ensuring consistency between the dataset and the model's output classes.

Each task is stored in the following format:

```python
{
    "text": "...",
    "entities": [(start, end, label), ...]
}
```

### Example

Input from Label Studio:

Text:

Barack Obama visited Paris.

Annotations:

- Barack Obama → PER
- Paris → LOC

Converted representation:

```python
{
    "text": "Barack Obama visited Paris.",
    "entities": [
        (0, 12, "PER"),
        (21, 26, "LOC")
    ]
}
```

This structured format serves as the foundation for the next preprocessing steps, where the text will be tokenized and entity labels will be converted into BIO tags for model training.

Finally, the number of loaded tasks and example samples are displayed to verify that the dataset has been parsed correctly.

In [6]:
# Parses a Label Studio JSON export into [{"text": ..., "entities": [(start, end, label), ...]}]
def parse_label_studio(path):
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    tasks = []
    skipped = 0
    seen_texts = set()

    for item in raw:
        text = item.get("data", {}).get("text", "").strip()
        if not text or text in seen_texts:
            skipped += 1
            continue
        seen_texts.add(text)

        entities = []
        annotations = item.get("annotations", [])
        if annotations:
            for result in annotations[0].get("result", []):
                val = result.get("value", {})
                start = val.get("start")
                end = val.get("end")
                labels = val.get("labels", [])
                if start is None or end is None or not labels:
                    continue
                label = labels[0]
                if label in ENTITY_TAGS:
                    entities.append((start, end, label))

        tasks.append({"text": text, "entities": entities})

    print(f"[Parse] Loaded {len(tasks)} tasks, skipped {skipped} duplicates/empty.")
    return tasks

tasks = parse_label_studio(DATA_PATH)
print(f"\nTotal tasks loaded: {len(tasks)}")
print(f"Example task text     : {tasks[0]['text'][:120]}...")
print(f"Example task entities : {tasks[0]['entities'][:5]}")


[Parse] Loaded 24044 tasks, skipped 822 duplicates/empty.

Total tasks loaded: 24044
Example task text     : In the 2024 presidential election , you could vote for Kamala Harris on the Democratic line or on the Working Families P...
Example task entities : [(7, 11, 'DATE'), (55, 68, 'PER'), (76, 86, 'NORP'), (102, 124, 'ORG')]


## Entity Distribution Analysis

This section analyzes the distribution of entity annotations across the entire dataset. It computes the frequency of each entity type, the number of sentences containing each tag, and the number of entities present in each sentence.

### Dataset Statistics

The following statistics are calculated:

- **Total entity spans** – Total number of annotated entities in the dataset.
- **Average spans per sentence** – Average number of entities present in each sentence.
- **Sentences with no entities** – Number of sentences containing only non-entity tokens.
- **Maximum entities in a sentence** – Largest number of entities appearing in a single sentence.

### Tag Frequency Analysis

For each entity type, the following information is reported:

- **Spans** – Total occurrences of the entity.
- **Percentage of all entities** – Relative frequency of the entity in the dataset.
- **Sentences with the tag** – Number of sentences containing at least one occurrence of that entity.

Rare entity categories defined in `RARE_TAGS` are highlighted to identify class imbalance.

### Why is this analysis important?

Entity distributions are often highly imbalanced. Some entity types may have thousands of examples, while others appear only a few times. Such imbalance can cause the model to favor frequent classes and perform poorly on underrepresented entities.

Analyzing the dataset helps:

- Understand the composition of the training data.
- Detect class imbalance.
- Identify rare entity categories.
- Guide decisions such as class weighting or data augmentation.
- Estimate the difficulty of learning specific entity types.

These statistics provide an overview of the dataset and help ensure that the model is trained on well-understood and properly balanced data.

In [7]:
# How many of each of the 17 tags exist across the whole dataset
tag_counts = Counter()
sentences_per_tag = Counter()
entities_per_sentence = []

for t in tasks:
    seen_here = set()
    entities_per_sentence.append(len(t["entities"]))
    for _, _, label in t["entities"]:
        tag_counts[label] += 1
        seen_here.add(label)
    for tag in seen_here:
        sentences_per_tag[tag] += 1

total_entities = sum(tag_counts.values())
print(f"Total entity spans      : {total_entities}")
print(f"Avg spans / sentence    : {total_entities/len(tasks):.2f}")
print(f"Sentences w/ 0 entities : {sum(1 for n in entities_per_sentence if n == 0)}")
print(f"Max entities in one sentence : {max(entities_per_sentence)}")

print(f"\n{'Tag':8s} {'Spans':>8s} {'% of all':>9s} {'Sentences w/ tag':>17s}")
print("-" * 48)
for tag in ENTITY_TAGS:
    c = tag_counts.get(tag, 0)
    pct = 100 * c / total_entities if total_entities else 0
    s = sentences_per_tag.get(tag, 0)
    flag = "  <- RARE" if tag in RARE_TAGS else ""
    print(f"{tag:8s} {c:8d} {pct:8.2f}% {s:17d}{flag}")


Total entity spans      : 70186
Avg spans / sentence    : 2.92
Sentences w/ 0 entities : 1581
Max entities in one sentence : 40

Tag         Spans  % of all  Sentences w/ tag
------------------------------------------------
PER         11385    16.22%              7947
ORG         14081    20.06%              9204
LOC         13215    18.83%              8396
FAC           602     0.86%               540  <- RARE
DATE         9356    13.33%              7279
TIME          848     1.21%               787  <- RARE
EVT           983     1.40%               933  <- RARE
PROD         1394     1.99%              1164
MONEY        2248     3.20%              1687
PCT          1593     2.27%              1151
LAW           721     1.03%               603
NORP         5061     7.21%              3807
LANG          202     0.29%               151  <- RARE
WOA          1403     2.00%              1241  <- RARE
QTY          1164     1.66%               919  <- RARE
ORD          1221     1.74%     

## Sentence-Level Entity Analysis

This section examines the distribution of entity annotations within an individual sentence. For a selected example, it counts the occurrences of each entity type and reports the total number of entities present.

### Purpose

Analyzing entities at the sentence level helps understand how annotations are distributed within individual samples. It provides insight into the complexity of sentences and verifies that entity labels have been assigned correctly.

### Output

For the selected sentence, the following information is displayed:

- **Text** – The original sentence.
- **Entity count by tag** – Number of entities belonging to each category.
- **Total entities in the sentence** – Total number of annotated entities.

By default, the code selects a sentence containing at least three entities. If no such sentence exists, the first sentence in the dataset is used. This provides a representative example for manual inspection and validation of the annotation quality.

In [8]:
# Example: how many of each tag appear in ONE sentence
def show_sentence_tags(task):
    c = Counter(label for _, _, label in task["entities"])
    print(f"Text: {task['text']}")
    print(f"Entity count by tag: {dict(c)}")
    print(f"Total entities in this sentence: {sum(c.values())}")

example = next((t for t in tasks if len(t["entities"]) >= 3), tasks[0])
show_sentence_tags(example)


Text: In the 2024 presidential election , you could vote for Kamala Harris on the Democratic line or on the Working Families Party line .
Entity count by tag: {'DATE': 1, 'PER': 1, 'NORP': 1, 'ORG': 1}
Total entities in this sentence: 4


## Dataset Splitting

This section randomly shuffles the dataset and divides it into three subsets: training, validation, and test sets.

### Training Set

The training set is used to optimize the model parameters. During training, the model learns patterns and relationships between tokens and entity labels.

### Validation Set

The validation set is used to monitor model performance during training. It helps tune hyperparameters and supports early stopping to prevent overfitting. The model never learns directly from this data.

### Test Set

The test set is completely held out and is not used during training or validation. It provides an unbiased estimate of the model's final performance on unseen data.

### Why shuffle the data?

Shuffling ensures that samples are randomly distributed among the three subsets. This reduces potential bias caused by the original ordering of the dataset and helps each subset better represent the overall data distribution.

### Why split the dataset?

Separating the data prevents information leakage and allows the model to be evaluated fairly on unseen examples.

### Why not use 5-Fold Cross Validation?

Although 5-fold cross validation provides multiple performance estimates, it requires training the model five times, which significantly increases computational cost. Using a single train-validation-test split is computationally more efficient and allows the held-out test set to provide a single, unbiased F1-score for evaluating the final model.

In [9]:
random.shuffle(tasks)

n_total = len(tasks)
n_test = int(n_total * TEST_SPLIT)
test_tasks = tasks[:n_test]
remaining = tasks[n_test:]

n_val = int(len(remaining) * VAL_SPLIT)
val_tasks = remaining[:n_val]
train_tasks = remaining[n_val:]

print(f"Total            : {n_total}")
print(f"Train            : {len(train_tasks)} ({100*len(train_tasks)/n_total:.1f}%)")
print(f"Validation       : {len(val_tasks)} ({100*len(val_tasks)/n_total:.1f}%)")
print(f"Test (held-out)  : {len(test_tasks)} ({100*len(test_tasks)/n_total:.1f}%)")
print()
print("5-fold CV skipped. This single split is faster, and the held-out test set")
print("below gives one clean F1 instead of 5 separate fold scores to interpret.")


Total            : 24044
Train            : 19476 (81.0%)
Validation       : 2164 (9.0%)
Test (held-out)  : 2404 (10.0%)

5-fold CV skipped. This single split is faster, and the held-out test set
below gives one clean F1 instead of 5 separate fold scores to interpret.


## Tokenizer Initialization

This section loads the tokenizer associated with the selected pretrained model. The tokenizer converts raw text into tokens and numerical IDs that can be processed by the transformer model.

### Tokenizer Information

The following properties are displayed:

- **Tokenizer class** – Type of tokenizer used by the model.
- **Vocabulary size** – Number of tokens recognized by the tokenizer.
- **Model maximum length** – Maximum sequence length supported by the pretrained model.
- **Training sequence length** – The `MAX_LENGTH` chosen for this experiment.

### Why is a tokenizer needed?

Transformer models cannot process raw text directly. Therefore, sentences must first be broken into smaller units called tokens and then converted into numerical IDs.

For example:

```text
Barack Obama visited Paris.
```

may be tokenized as:

```text
["Barack", "Obama", "visited", "Paris", "."]
```

and converted into token IDs before being fed to the model.

### Why use the tokenizer corresponding to the pretrained model?

Each pretrained model has its own vocabulary and tokenization rules. Using the matching tokenizer ensures that the input representation is consistent with the data on which the model was originally pretrained.

### Why check the maximum length?

Transformer models have a limited context window. The chosen `MAX_LENGTH` determines the maximum number of tokens retained from each sentence. Sequences longer than this value are truncated, while shorter sequences are padded. Selecting an appropriate sequence length helps balance memory usage and contextual information.


In [10]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

print(f"Tokenizer class       : {tokenizer.__class__.__name__}")
print(f"Vocab size            : {tokenizer.vocab_size}")
print(f"Tokenizer's own max   : {tokenizer.model_max_length}")
print(f"Our MAX_LENGTH        : {MAX_LENGTH}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer class       : BertTokenizer
Vocab size            : 30522
Tokenizer's own max   : 512
Our MAX_LENGTH        : 256


## Sequence Length Analysis

This section analyzes the distribution of tokenized sentence lengths before applying truncation. The number of tokens in each sentence is measured using the tokenizer, including special tokens such as `[CLS]` and `[SEP]`.

### Statistics Computed

The following statistics are reported:

- **Mean length** – Average number of tokens per sentence.
- **Median length** – Middle value of the token length distribution.
- **95th percentile** – Length below which 95% of sentences fall.
- **Maximum length** – Length of the longest sentence in the dataset.

In addition, the number and percentage of sentences exceeding 128 and 256 tokens are calculated.

### Why is this analysis important?

Transformer models require a fixed maximum sequence length. Sentences longer than this limit are truncated, causing some tokens and their corresponding entity labels to be discarded.

By examining the token length distribution, we can:

- Understand the characteristics of the dataset.
- Estimate how much information would be lost due to truncation.
- Choose an appropriate value for `MAX_LENGTH`.
- Balance contextual information and GPU memory requirements.

### Why increase `MAX_LENGTH` from 128 to 256?

With `MAX_LENGTH = 128`, all sentences longer than 128 tokens are truncated, potentially removing important entities appearing near the end of the sentence.

Increasing the sequence length to 256 allows the model to preserve more context and retain entities that would otherwise be discarded. However, longer sequences require more memory and increase training time.

### Interpretation

- Sentences with length ≤ 128 are fully preserved at the old setting.
- Sentences with length between 129 and 256 are preserved after increasing `MAX_LENGTH` to 256.
- Sentences longer than 256 tokens are still truncated and may lose information.

This analysis helps determine whether the chosen sequence length provides a good trade-off between model performance and computational cost.

In [11]:
# How many sentences actually exceed 128 / 256 tokens, before any truncation
lengths = np.array([
    len(tokenizer(t["text"], truncation=False)["input_ids"]) for t in tasks
])

print("Token length stats (includes [CLS]/[SEP], no truncation applied):")
print(f"  mean   : {lengths.mean():.1f}")
print(f"  median : {np.median(lengths):.1f}")
print(f"  95th % : {np.percentile(lengths, 95):.1f}")
print(f"  max    : {lengths.max()}")
print()
print(f"  sentences > 128 tokens : {(lengths > 128).sum():4d}  ({100*(lengths>128).mean():.2f}%)  "
      f"<- these were getting cut off / discarded at the old MAX_LENGTH=128")
print(f"  sentences > 256 tokens : {(lengths > 256).sum():4d}  ({100*(lengths>256).mean():.2f}%)  "
      f"<- still truncated even at the new MAX_LENGTH=256")


Token length stats (includes [CLS]/[SEP], no truncation applied):
  mean   : 31.9
  median : 29.0
  95th % : 62.0
  max    : 202

  sentences > 128 tokens :   18  (0.07%)  <- these were getting cut off / discarded at the old MAX_LENGTH=128
  sentences > 256 tokens :    0  (0.00%)  <- still truncated even at the new MAX_LENGTH=256


## Tokenization and Label Alignment

This section converts raw text into token IDs and aligns entity annotations with the tokenized representation using the BIO tagging scheme.

### Tokenization

The tokenizer transforms each sentence into a sequence of tokens and corresponding numerical IDs that can be processed by the transformer model. Sequences are truncated to `MAX_LENGTH` and padded when necessary to ensure a fixed input size.

### Offset Mapping

For every token, the tokenizer returns its character start and end positions in the original text. These offsets are used to determine whether a token belongs to an annotated entity span.

### BIO Label Assignment

Entity labels are converted into the BIO format:

- **B-** indicates the beginning of an entity.
- **I-** indicates the continuation of the same entity.
- **O** indicates tokens that do not belong to any entity.

For example:

Text:

```
Barack Obama visited Paris.
```

Entities:

```
(0, 12, PER)
(21, 26, LOC)
```

Tokenized sequence:

| Token | Label |
|---------|------|
| Barack | B-PER |
| Obama | I-PER |
| visited | O |
| Paris | B-LOC |
| . | O |

### Handling Subword Tokens

Transformer tokenizers may split a word into multiple subword tokens. In such cases:

- The first subword receives a `B-` label if it starts the entity.
- Subsequent subwords receive `I-` labels.

This ensures that entity boundaries are preserved even after word splitting.

### Ignoring Special Tokens

Special tokens such as `[CLS]`, `[SEP]`, and padding tokens do not correspond to actual words. These positions are assigned the label `-100`, which instructs the loss function to ignore them during training.

### Output

For each sentence, three arrays are generated:

- **input_ids** – Token IDs representing the sentence.
- **attention_mask** – Indicates which positions correspond to real tokens and which are padding.
- **labels** – BIO labels aligned with each token.

These aligned token-label pairs constitute the final input used for training the Named Entity Recognition model.

### Verification

A sample sentence is processed and printed to display:

- The original text.
- The annotated entities.
- The number of non-padding tokens.
- Each token together with its assigned label.

This serves as a sanity check to verify that tokenization and label alignment have been performed correctly before training begins.

In [12]:
# Tokenizes text and aligns BIO labels to subword tokens
def tokenize_and_align(task_list, tokenizer, label2id, max_length=MAX_LENGTH):
    encoded = []
    for task in task_list:
        text = task["text"]
        entities = sorted(task["entities"], key=lambda x: x[0])

        tok = tokenizer(
            text,
            max_length=max_length,
            truncation=True,
            padding="max_length",
            return_offsets_mapping=True,
            return_tensors=None,
        )

        offsets = tok["offset_mapping"]
        word_ids = tok.word_ids()

        labels = []
        prev_word_id = None

        for offset, wid in zip(offsets, word_ids):
            if wid is None:
                labels.append(-100)
                prev_word_id = wid
                continue

            char_start, char_end = offset
            matched_label = "O"
            matched_entity_start = None
            for (ent_start, ent_end, ent_label) in entities:
                if char_start >= ent_start and char_end <= ent_end and char_end > char_start:
                    matched_label = ent_label
                    matched_entity_start = ent_start
                    break

            if matched_label != "O":
                if wid != prev_word_id:
                    bio_label = f"B-{matched_label}" if char_start == matched_entity_start else f"I-{matched_label}"
                else:
                    bio_label = f"I-{matched_label}"
            else:
                bio_label = "O"

            labels.append(label2id.get(bio_label, label2id["O"]))
            prev_word_id = wid

        encoded.append({
            "input_ids": tok["input_ids"],
            "attention_mask": tok["attention_mask"],
            "labels": labels,
        })
    return encoded

# Verbose demo: see exactly how one sentence becomes tokens + labels fed to the model
demo_task = next((t for t in tasks if len(t["entities"]) >= 2), tasks[0])
demo_enc = tokenize_and_align([demo_task], tokenizer, label2id, max_length=MAX_LENGTH)[0]

demo_tokens = tokenizer.convert_ids_to_tokens(demo_enc["input_ids"])
real_len = sum(demo_enc["attention_mask"])

print(f"Text     : {demo_task['text']}")
print(f"Entities : {demo_task['entities']}")
print(f"\nReal (non-padding) tokens fed to model: {real_len} / {MAX_LENGTH}\n")
print(f"{'#':>3s}  {'Token':20s} {'Label'}")
print("-" * 40)
for i, (tok, lab_id) in enumerate(zip(demo_tokens, demo_enc["labels"])):
    lab = "[IGNORED]" if lab_id == -100 else id2label[lab_id]
    if tok == "[PAD]" and i > real_len + 2:
        print("...  (padding continues) ...")
        break
    print(f"{i:3d}  {tok:20s} {lab}")


Text     : Brigadier General Vince Brooks confirmed a USA Warthog aircraft was shot down by an Iraqi missile today .
Entities : [(18, 30, 'PER'), (43, 54, 'PROD'), (84, 89, 'NORP'), (98, 103, 'DATE')]

Real (non-padding) tokens fed to model: 22 / 256

  #  Token                Label
----------------------------------------
  0  [CLS]                [IGNORED]
  1  brigadier            O
  2  general              O
  3  vince                B-PER
  4  brooks               I-PER
  5  confirmed            O
  6  a                    O
  7  usa                  B-PROD
  8  war                  I-PROD
  9  ##th                 I-PROD
 10  ##og                 I-PROD
 11  aircraft             O
 12  was                  O
 13  shot                 O
 14  down                 O
 15  by                   O
 16  an                   O
 17  iraqi                B-NORP
 18  missile              O
 19  today                B-DATE
 20  .                    O
 21  [SEP]                [IGNORED]
 22  

## Dataset Tokenization

This section applies tokenization and label alignment to the training, validation, and test sets. Each sentence is converted into token IDs, attention masks, and BIO labels that are compatible with the transformer model.

### Training Set

The training data is tokenized and prepared for model learning.

### Validation Set

The validation data is processed in the same manner and is used to monitor performance during training.

### Test Set

The test data is tokenized separately and remains untouched until the final evaluation stage.

### Fixed-Length Sequences

All sequences are padded or truncated to `MAX_LENGTH`, ensuring that every sample has the same length. This enables efficient batch processing and compatibility with the transformer architecture.

### Output Format

Each encoded sample contains:

- **input_ids** – Numerical representation of tokens.
- **attention_mask** – Indicates which positions correspond to actual tokens and which are padding.
- **labels** – BIO labels aligned with the tokenized sequence.

Finally, the number of encoded samples in each split and the sequence length of each sample are displayed to verify that preprocessing has been performed correctly.

In [13]:
print("Tokenizing train set...")
encoded_train = tokenize_and_align(train_tasks, tokenizer, label2id)
print("Tokenizing val set...")
encoded_val = tokenize_and_align(val_tasks, tokenizer, label2id)
print("Tokenizing test set...")
encoded_test = tokenize_and_align(test_tasks, tokenizer, label2id)

print(f"\nEncoded train : {len(encoded_train)}")
print(f"Encoded val   : {len(encoded_val)}")
print(f"Encoded test  : {len(encoded_test)}")
print(f"Each sample input_ids length: {len(encoded_train[0]['input_ids'])} (should equal MAX_LENGTH={MAX_LENGTH})")


Tokenizing train set...
Tokenizing val set...
Tokenizing test set...

Encoded train : 19476
Encoded val   : 2164
Encoded test  : 2404
Each sample input_ids length: 256 (should equal MAX_LENGTH=256)


## Tag Presence Verification

This section verifies that each entity type is represented in the tokenized training, validation, and test sets. After tokenization and BIO label assignment, the number of labeled tokens corresponding to each entity category is counted.

### Purpose

The goal of this analysis is to ensure that all entity types are present in the dataset splits, particularly in the training set. A model can only learn to recognize an entity type if it has seen examples of that type during training.

### Token-Level Statistics

For each split, the number of tokens belonging to each entity category is computed. Special tokens and padding positions labeled with `-100` are ignored.

The resulting statistics show:

- The number of entity tokens for each tag.
- Whether a tag is present or absent in a particular split.
- The distribution of entity labels across training, validation, and test sets.

### Importance of Training Coverage

If an entity type has zero examples in the training set, the model will never learn to predict that class, regardless of how many examples appear in the validation or test sets.

For example:

| Tag | Training Tokens |
|------|----------------|
| PER | 4500 |
| ORG | 3200 |
| LOC | 2800 |
| LANG | 0 |

In this case, the model cannot learn the `LANG` entity because no training examples are available.

### Why is this check important?

- Ensures that every entity type is represented in the training data.
- Detects missing classes before training begins.
- Prevents misleading evaluation results.
- Helps identify severe class imbalance.
- Confirms that tokenization and label alignment have preserved all entity types correctly.

Finally, the code reports whether all entity categories have at least one training example. If a tag is completely absent from the training set, a warning is displayed because the model will be unable to learn that entity type.

In [14]:
# Confirms every one of the 17 tags has real training examples after tokenization
def tag_presence_report(encoded, name):
    counter = Counter()
    for sample in encoded:
        for lab_id in sample["labels"]:
            if lab_id == -100:
                continue
            lab = id2label[lab_id]
            if lab != "O":
                tag = lab.split("-", 1)[1]
                counter[tag] += 1
    print(f"\n{name} set — token counts per tag:")
    for tag in ENTITY_TAGS:
        c = counter.get(tag, 0)
        warn = "  *** ZERO -- model will NEVER see this tag in this split ***" if c == 0 else ""
        print(f"  {tag:8s}: {c:6d}{warn}")
    return counter

train_tag_counts = tag_presence_report(encoded_train, "TRAIN")
val_tag_counts = tag_presence_report(encoded_val, "VAL")
test_tag_counts = tag_presence_report(encoded_test, "TEST")

missing_in_train = [t for t in ENTITY_TAGS if train_tag_counts.get(t, 0) == 0]
if missing_in_train:
    print(f"\n*** WARNING: zero training examples for: {missing_in_train} -- model cannot learn these. ***")
else:
    print("\nAll 17 tags have at least one training example.")



TRAIN set — token counts per tag:
  PER     :  25411
  ORG     :  33307
  LOC     :  18306
  FAC     :   1586
  DATE    :  16378
  TIME    :   2041
  EVT     :   2898
  PROD    :   3197
  MONEY   :   7351
  PCT     :   4077
  LAW     :   2311
  NORP    :   4934
  LANG    :    178
  WOA     :   4892
  QTY     :   3555
  ORD     :    998
  CARD    :   6539

VAL set — token counts per tag:
  PER     :   2749
  ORG     :   3481
  LOC     :   2123
  FAC     :    212
  DATE    :   1721
  TIME    :    265
  EVT     :    363
  PROD    :    341
  MONEY   :    901
  PCT     :    483
  LAW     :    221
  NORP    :    500
  LANG    :     30
  WOA     :    601
  QTY     :    339
  ORD     :    116
  CARD    :    777

TEST set — token counts per tag:
  PER     :   3014
  ORG     :   4280
  LOC     :   2237
  FAC     :    187
  DATE    :   2032
  TIME    :    220
  EVT     :    315
  PROD    :    422
  MONEY   :    888
  PCT     :    465
  LAW     :    279
  NORP    :    645
  LANG    :     28
  WOA

## Creating PyTorch Datasets

This section defines a custom `NERDataset` class to make the tokenized data compatible with PyTorch and the Hugging Face `Trainer`.

### Dataset Structure

Each sample consists of three components:

- **input_ids** – Numerical representation of the tokenized text.
- **attention_mask** – Indicates which positions correspond to real tokens and which are padding.
- **labels** – BIO labels aligned with the token sequence.

These components are converted into PyTorch tensors of type `torch.long`, which are required for model training.

### Dataset Interface

The `NERDataset` class implements two essential methods:

- **`__len__()`** – Returns the total number of samples in the dataset.
- **`__getitem__()`** – Retrieves a single sample and converts it into tensor format.

### Dataset Splits

Three dataset objects are created:

- **train_ds** – Used to train the model.
- **val_ds** – Used to monitor performance and support early stopping.
- **test_ds** – Reserved for final evaluation on unseen data.

### Why is a Dataset class needed?

Although the tokenized samples are stored as Python dictionaries, PyTorch models expect tensors rather than lists. The custom dataset class provides an efficient interface for accessing and batching samples during training.

### Output Verification

The code displays:

- The number of samples in each dataset split.
- The shape of the tensors corresponding to a sample.

For example, with `MAX_LENGTH = 256`, a sample may have:

- `input_ids.shape = (256,)`
- `labels.shape = (256,)`

confirming that every sequence has a fixed length and is ready to be fed into the transformer model.

This dataset abstraction forms the bridge between the preprocessing stage and the model training pipeline.

In [15]:
class NERDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        return {
            "input_ids": torch.tensor(s["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(s["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(s["labels"], dtype=torch.long),
        }

train_ds = NERDataset(encoded_train)
val_ds = NERDataset(encoded_val)
test_ds = NERDataset(encoded_test)

print(f"train_ds: {len(train_ds)} | val_ds: {len(val_ds)} | test_ds: {len(test_ds)}")
sample = train_ds[0]
print(f"Sample shapes -> input_ids: {sample['input_ids'].shape}, labels: {sample['labels'].shape}")


train_ds: 19476 | val_ds: 2164 | test_ds: 2404
Sample shapes -> input_ids: torch.Size([256]), labels: torch.Size([256])


## Class Weighting for Rare Entities

This section assigns higher loss weights to underrepresented entity types in order to mitigate class imbalance. Rare entity categories are given greater importance during training by increasing their contribution to the loss function.

### Motivation

Entity distributions are often imbalanced, with some tags appearing much less frequently than others. Without corrective measures, the model may become biased toward common entity types and perform poorly on rare classes.

### Weight Assignment

A weight vector is created with one entry for each label class. By default, every class receives a weight of 1.0. Labels corresponding to entity types listed in `RARE_TAGS` are assigned a higher weight specified by `RARE_WEIGHT`.

Since BIO tagging is used, both the `B-` and `I-` versions of a rare entity receive the same increased weight.

For example, if:

- `RARE_WEIGHT = 3.0`

then:

- `B-WOA` → 3.0
- `I-WOA` → 3.0
- `B-EVT` → 3.0
- `I-EVT` → 3.0

while common labels such as:

- `O`
- `B-PER`
- `I-PER`

retain the default weight of 1.0.

### Why is this necessary?

Without class weighting, the loss function treats every label equally. Because common entities appear much more frequently, they dominate the optimization process, causing the model to pay relatively little attention to rare classes.

By increasing the loss associated with rare entities, mistakes made on these classes become more costly, encouraging the model to learn better representations for underrepresented categories.

### Benefits

- Reduces the effect of class imbalance.
- Improves learning for rare entity types.
- Prevents the model from focusing only on frequent classes.
- Encourages more balanced performance across all entity categories.

Finally, the weight assigned to each label is printed to verify that rare entity classes have received the intended emphasis during training.

In [16]:
def build_class_weights(label2id):
    num_labels = len(label2id)
    weights = torch.ones(num_labels)
    for label, idx in label2id.items():
        for rare in RARE_TAGS:
            if rare in label:
                weights[idx] = RARE_WEIGHT
    return weights

class_weights = build_class_weights(label2id)
print("Per-label loss weight (rare tags get 3x):")
for i, w in enumerate(class_weights):
    print(f"  {id2label[i]:10s}: {w.item():.1f}")


Per-label loss weight (rare tags get 3x):
  O         : 1.0
  B-PER     : 1.0
  I-PER     : 1.0
  B-ORG     : 1.0
  I-ORG     : 1.0
  B-LOC     : 1.0
  I-LOC     : 1.0
  B-FAC     : 3.0
  I-FAC     : 3.0
  B-DATE    : 1.0
  I-DATE    : 1.0
  B-TIME    : 3.0
  I-TIME    : 3.0
  B-EVT     : 3.0
  I-EVT     : 3.0
  B-PROD    : 1.0
  I-PROD    : 1.0
  B-MONEY   : 1.0
  I-MONEY   : 1.0
  B-PCT     : 1.0
  I-PCT     : 1.0
  B-LAW     : 1.0
  I-LAW     : 1.0
  B-NORP    : 1.0
  I-NORP    : 1.0
  B-LANG    : 3.0
  I-LANG    : 3.0
  B-WOA     : 3.0
  I-WOA     : 3.0
  B-QTY     : 3.0
  I-QTY     : 3.0
  B-ORD     : 3.0
  I-ORD     : 3.0
  B-CARD    : 3.0
  I-CARD    : 3.0


## Custom Trainer and Evaluation Metrics

This section defines a custom trainer with weighted loss and specifies the evaluation metrics used to assess model performance.

### Weighted Loss Function

A custom `WeightedTrainer` class extends the Hugging Face `Trainer` by overriding the `compute_loss()` method. Instead of using the default loss function, a weighted cross-entropy loss is employed.

The class weights computed previously are incorporated into the loss function so that errors on rare entity types contribute more strongly to the optimization process.

Special tokens and padding positions labeled with `-100` are ignored during loss computation.

### Why use a weighted loss?

Entity distributions are often imbalanced. Frequent entity types may dominate the learning process, causing poor performance on rare classes. Weighted loss assigns greater importance to rare entities and helps achieve more balanced predictions.

### Cross-Entropy Loss

The model outputs logits representing scores for each label class. These logits are compared with the true labels using cross-entropy loss, which measures the discrepancy between predictions and ground truth.

The model then updates its parameters to minimize this loss.

### Evaluation Metrics

The `compute_metrics()` function calculates the model's performance on the validation and test sets.

Predicted label IDs are converted into label names using `id2label`. Positions labeled with `-100` are ignored because they correspond to special tokens and padding.

Three metrics are reported:

- **Precision** – Measures how many predicted entities are correct.

  Precision = Correct Predictions / Total Predicted Entities

- **Recall** – Measures how many true entities are successfully identified.

  Recall = Correct Predictions / Total Actual Entities

- **F1-score** – Harmonic mean of precision and recall, providing a balanced measure of performance.

### Why use F1-score?

Named Entity Recognition datasets are typically imbalanced, making accuracy an unreliable metric. F1-score considers both precision and recall and is therefore the standard metric for evaluating NER systems.

### Benefits of this approach

- Incorporates class weights to improve learning for rare entities.
- Ignores padding and special tokens during optimization.
- Provides meaningful evaluation metrics for NER.
- Enables fair assessment of the model's ability to identify entities.

Together, the custom trainer and metric computation framework ensure that the model is optimized effectively and evaluated using standard Named Entity Recognition metrics.

In [17]:
class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = self.class_weights.to(logits.device)
        loss_fn = nn.CrossEntropyLoss(weight=weights, ignore_index=-100)
        loss = loss_fn(logits.view(-1, logits.shape[-1]), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def make_compute_metrics(id2label):
    def compute_metrics(p):
        preds, labels = p
        preds = np.argmax(preds, axis=2)
        true_labels_list, pred_labels_list = [], []
        for pred_seq, label_seq in zip(preds, labels):
            true_seq, pred_seq_ = [], []
            for p_id, l_id in zip(pred_seq, label_seq):
                if l_id == -100:
                    continue
                true_seq.append(id2label[l_id])
                pred_seq_.append(id2label[p_id])
            true_labels_list.append(true_seq)
            pred_labels_list.append(pred_seq_)
        return {
            "f1": f1_score(true_labels_list, pred_labels_list),
            "precision": precision_score(true_labels_list, pred_labels_list),
            "recall": recall_score(true_labels_list, pred_labels_list),
        }
    return compute_metrics

print("WeightedTrainer and compute_metrics defined.")


WeightedTrainer and compute_metrics defined.


## Model Initialization

This section loads the pretrained transformer model and adapts it for the Named Entity Recognition (NER) task.

### Loading the Pretrained Model

The model is initialized from the selected backbone specified by `BASE_MODEL`. Since the pretrained model may have been trained on a different set of entity labels, the classification head is reconfigured to match the number of labels in the current dataset.

### Label Configuration

The following mappings are provided to the model:

- **num_labels** – Total number of output classes.
- **label2id** – Maps entity labels to numerical IDs.
- **id2label** – Converts predicted IDs back into human-readable labels.

These mappings ensure consistency between the dataset and the model's predictions.

### Handling Different Output Sizes

The parameter `ignore_mismatched_sizes=True` allows the classification layer to be automatically resized when the number of labels in the current task differs from that of the original pretrained model.

As a result, the transformer encoder retains its pretrained knowledge, while a new classifier head is created for the target entity categories.

### Model Information

The following information is displayed:

- **Model name** – The pretrained backbone being used.
- **Number of output classes** – Corresponds to the total number of BIO labels.
- **Total parameters** – Number of trainable parameters in the network, expressed in millions.

### Why use a pretrained model?

Training a transformer from scratch requires enormous amounts of data and computational resources. By starting from a pretrained model, the network already possesses rich linguistic knowledge acquired from large corpora.

Fine-tuning enables the model to adapt this knowledge to the specific Named Entity Recognition task using a comparatively small annotated dataset.

Thus, this step initializes the neural network and prepares it for task-specific training.

In [18]:
model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

print(f"Model          : {BASE_MODEL}")
print(f"Output classes : {model.config.num_labels}")
print(f"Total params   : {sum(p.numel() for p in model.parameters())/1e6:.1f}M")


pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForTokenClassification LOAD REPORT from: google/electra-base-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
electra.embeddings_project.weight                 | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
electra.embeddings_project.bias                   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.weight                                 | MISSING    | 
classifier.bias                                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream t

Model          : google/electra-base-discriminator
Output classes : 35
Total params   : 108.9M


## Training Configuration

This section defines the hyperparameters and training settings used for fine-tuning the NER model. These parameters control how the model learns, evaluates performance, and saves checkpoints.

### Training Parameters

- **Number of Epochs (`num_train_epochs`)**

  Specifies how many times the model iterates over the entire training dataset. Multiple epochs allow the model to gradually improve its understanding of entity patterns.

- **Batch Size (`per_device_train_batch_size`)**

  Determines how many samples are processed simultaneously in one forward pass. Smaller batch sizes reduce memory usage but may lead to noisier gradient updates.

- **Gradient Accumulation (`gradient_accumulation_steps`)**

  Gradients are accumulated over several mini-batches before updating the model parameters. This simulates a larger effective batch size while keeping GPU memory requirements manageable.

  Effective Batch Size = Batch Size × Gradient Accumulation Steps

- **Learning Rate (`learning_rate`)**

  Controls the magnitude of parameter updates. A small learning rate allows gradual adaptation of pretrained weights and helps prevent unstable training.

- **Weight Decay (`weight_decay`)**

  Acts as a regularization technique to reduce overfitting by discouraging excessively large model weights.

- **Warmup Ratio (`warmup_ratio`)**

  Gradually increases the learning rate during the initial phase of training. This stabilizes optimization and prevents large parameter updates at the beginning.

### Evaluation and Checkpointing

- **Evaluation Strategy (`eval_strategy="epoch"`)**

  The model is evaluated after every epoch to monitor performance.

- **Save Strategy (`save_strategy="epoch"`)**

  A checkpoint is saved after each epoch, allowing intermediate models to be preserved.

- **Load Best Model at End**

  After training, the checkpoint with the highest validation F1-score is automatically restored.

- **Metric for Best Model**

  The F1-score is used to determine which checkpoint performs best.

### Mixed Precision Training

If a GPU is available, half-precision floating-point arithmetic (`fp16`) is enabled.

Benefits include:

- Reduced GPU memory consumption.
- Faster training.
- Ability to train larger models and longer sequences.

### Logging

Training progress is recorded every 50 steps, allowing monitoring of loss and optimization behavior.

### Reproducibility

The random seed is fixed to ensure consistent experimental results across multiple runs.

### Reporting

External logging services are disabled (`report_to="none"`), so training statistics are stored locally.

Overall, these hyperparameters control the optimization process and ensure that training is efficient, stable, and reproducible.

In [19]:
args = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "model"),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=max(BATCH_SIZE * 2, 4),
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    seed=SEED,
    report_to="none",
)

print(f"Effective batch size : {BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"FP16 enabled          : {args.fp16}")
print(f"Epochs                : {NUM_EPOCHS}")
print(f"Eval strategy         : {args.eval_strategy}")


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Effective batch size : 8 x 2 = 16
FP16 enabled          : True
Epochs                : 8
Eval strategy         : IntervalStrategy.EPOCH


## Model Training

This section initializes the custom `WeightedTrainer` and starts the fine-tuning process.

### Trainer Components

The trainer combines several components required for training:

- **Model** – The pretrained transformer adapted for token classification.
- **Training Arguments** – Define hyperparameters such as learning rate, batch size, number of epochs, and evaluation strategy.
- **Training Dataset** – Used to optimize the model parameters.
- **Validation Dataset** – Used to monitor performance during training.
- **Weighted Loss Function** – Gives greater importance to rare entity classes.
- **Evaluation Metrics** – Computes precision, recall, and F1-score after each evaluation.
- **Early Stopping Callback** – Stops training when the validation performance stops improving.

### Training Process

During each epoch, the model performs the following steps:

1. Processes batches from the training dataset.
2. Computes the weighted cross-entropy loss.
3. Updates the model parameters using backpropagation.
4. Evaluates performance on the validation set.
5. Calculates precision, recall, and F1-score.
6. Saves a checkpoint at the end of the epoch.

### Early Stopping

Training does not necessarily continue for all specified epochs. If the validation F1-score fails to improve for a certain number of consecutive evaluations (`EARLY_STOP_PAT`), training is terminated automatically.

This prevents unnecessary computation and reduces the risk of overfitting.

### Best Model Selection

Since `load_best_model_at_end=True` is enabled, the checkpoint with the highest validation F1-score is automatically restored after training. Therefore, the final model corresponds to the best-performing epoch rather than the last epoch.

### Why is this important?

The training pipeline combines optimization, evaluation, checkpointing, and early stopping into a single framework. This ensures that:

- The model learns from the training data.
- Performance is continuously monitored on unseen validation samples.
- Rare entity types receive additional attention through weighted loss.
- Overfitting is minimized.
- The best-performing model is retained for final evaluation.

The `trainer.train()` command then starts the complete fine-tuning process.

In [20]:
trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=make_compute_metrics(id2label),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PAT)],
)

trainer.train()


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.531763,0.237898,0.799058,0.767021,0.833888
2,0.353760,0.179933,0.828087,0.799380,0.858932
3,0.245512,0.174084,0.851838,0.832074,0.872563
4,0.192965,0.181429,0.854284,0.830440,0.879537
5,0.198972,0.191284,0.861807,0.841875,0.882707
6,0.130211,0.199386,0.866325,0.848927,0.884451
7,0.106071,0.208640,0.868221,0.851547,0.885560
8,0.091757,0.212054,0.870568,0.856223,0.885402


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=9744, training_loss=0.34779187718831456, metrics={'train_runtime': 2345.9193, 'train_samples_per_second': 66.417, 'train_steps_per_second': 4.154, 'total_flos': 2.036213578801152e+16, 'train_loss': 0.34779187718831456, 'epoch': 8.0})

In [21]:
val_metrics = trainer.evaluate()
print("\nValidation set metrics (best checkpoint):")
for k, v in val_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")


Training Loss,Validation Loss,Epoch,F1,Precision,Recall
0.091757,0.212054,8,0.870568,0.856223,0.885402



Validation set metrics (best checkpoint):
  eval_loss: 0.2121
  eval_f1: 0.8706
  eval_precision: 0.8562
  eval_recall: 0.8854


# ELECTRA-base-discriminator Held-Out Test Results

The fine-tuned ELECTRA-base-discriminator model was evaluated on a completely unseen test set to measure its ability to generalize to new data after training.

## Validation Performance

The best checkpoint selected during training achieved the following validation metrics:

| Metric          | Value  |
| --------------- | ------ |
| Validation Loss | 0.2121 |
| Precision       | 0.8562 |
| Recall          | 0.8854 |
| F1 Score        | 0.8706 |

The best model was automatically restored from the checkpoint with the highest validation F1-score.

## Overall Test Performance

| Metric            | Value  |
| ----------------- | ------ |
| Precision         | 0.8574 |
| Recall            | 0.8867 |
| Micro F1-score    | 0.8718 |
| Macro F1-score    | 0.8137 |
| Weighted F1-score | 0.8729 |

The held-out test F1-score of **0.8718** slightly exceeds the validation F1-score of **0.8706**, demonstrating strong generalization and no evidence of overfitting.

## Entity-wise Performance

The model achieved excellent results on frequently occurring entity types:

| Entity | F1 Score |
| ------ | -------- |
| PER    | 0.9403   |
| LOC    | 0.9279   |
| NORP   | 0.9224   |
| PCT    | 0.9115   |
| MONEY  | 0.9007   |
| CARD   | 0.8819   |
| DATE   | 0.8736   |
| ORD    | 0.8667   |
| LANG   | 0.8621   |

Strong performance was also obtained on medium-frequency entities:

| Entity | F1 Score |
| ------ | -------- |
| ORG    | 0.8220   |
| TIME   | 0.8229   |
| PROD   | 0.7575   |
| EVT    | 0.7551   |
| QTY    | 0.7363   |
| LAW    | 0.7051   |

Lower scores were observed for the rarest entity categories:

| Entity | F1 Score |
| ------ | -------- |
| WOA    | 0.5763   |
| FAC    | 0.5714   |

## Generalization Analysis

| Metric    | Validation | Test   |
| --------- | ---------- | ------ |
| Precision | 0.8562     | 0.8574 |
| Recall    | 0.8854     | 0.8867 |
| F1 Score  | 0.8706     | 0.8718 |

The extremely small difference between validation and test performance confirms that the model learned robust entity representations and generalized effectively to unseen examples.

## Conclusion

The ELECTRA-base-discriminator model achieved a final held-out test F1-score of **0.8718**, outperforming its validation F1-score of **0.8706**. The model demonstrated excellent recognition of major entity categories such as PER, LOC, NORP, MONEY, and PCT, while also achieving competitive performance on several low-frequency entity types including EVT and LAW. These results indicate that the model is well-balanced, stable, and capable of strong real-world Named Entity Recognition performance.


In [22]:
# This is the number that matters: a genuinely held-out test set the model
# never saw during training or early stopping.
preds_out = trainer.predict(test_ds)
preds = np.argmax(preds_out.predictions, axis=2)
labels = preds_out.label_ids

true_list, pred_list = [], []
for pred_seq, label_seq in zip(preds, labels):
    true_seq, pred_seq_ = [], []
    for p_id, l_id in zip(pred_seq, label_seq):
        if l_id == -100:
            continue
        true_seq.append(id2label[l_id])
        pred_seq_.append(id2label[p_id])
    true_list.append(true_seq)
    pred_list.append(pred_seq_)

report = classification_report(true_list, pred_list, digits=4)
overall_f1 = f1_score(true_list, pred_list)
overall_p = precision_score(true_list, pred_list)
overall_r = recall_score(true_list, pred_list)

print(report)
print(f"\nOverall Precision : {overall_p:.4f}")
print(f"Overall Recall    : {overall_r:.4f}")
print(f"Overall F1-score  : {overall_f1:.4f}   <- your real held-out test F1")

os.makedirs(OUTPUT_DIR, exist_ok=True)
report_path = os.path.join(OUTPUT_DIR, "final_test_report.txt")
with open(report_path, "w") as f:
    f.write(report)
    f.write(f"\nPrecision: {overall_p:.4f}\nRecall: {overall_r:.4f}\nF1-score: {overall_f1:.4f}\n")
print(f"\nSaved -> {report_path}")


              precision    recall  f1-score   support

        CARD     0.8574    0.9078    0.8819       477
        DATE     0.8503    0.8982    0.8736       904
         EVT     0.7184    0.7957    0.7551        93
         FAC     0.5060    0.6562    0.5714        64
        LANG     0.7812    0.9615    0.8621        26
         LAW     0.6250    0.8088    0.7051        68
         LOC     0.9283    0.9276    0.9279      1312
       MONEY     0.8831    0.9189    0.9007       222
        NORP     0.9165    0.9283    0.9224       544
         ORD     0.8254    0.9123    0.8667       114
         ORG     0.8188    0.8251    0.8220      1424
         PCT     0.9085    0.9145    0.9115       152
         PER     0.9287    0.9522    0.9403      1108
        PROD     0.7600    0.7550    0.7575       151
         QTY     0.7400    0.7327    0.7363       101
        TIME     0.7500    0.9114    0.8229        79
         WOA     0.5346    0.6250    0.5763       136

   micro avg     0.8574   

In [23]:
final_dir = os.path.join(OUTPUT_DIR, "final_model")
model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"Model + tokenizer saved -> {final_dir}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model + tokenizer saved -> ./ner_output/final_model


## Saving the Final Model

After training and evaluation, the best-performing model and its corresponding tokenizer are saved to disk for future use.

### Model Saving

The `save_pretrained()` method stores:

- The trained model weights.
- Model configuration files.
- Label mappings (`id2label` and `label2id`).

These files enable the model to be reloaded later without retraining.

### Tokenizer Saving

The tokenizer is saved alongside the model. This preserves:

- Vocabulary files.
- Tokenization rules.
- Special tokens and configuration settings.

Using the same tokenizer during inference is essential because the model expects inputs to be processed in exactly the same way as during training.

### Output Directory

Both the model and tokenizer are saved inside:

```
./ner_output/final_model
```

This directory contains all files required for deployment, inference, or further fine-tuning.


In [24]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Total tasks         : {n_total}")
print(f"Train / Val / Test  : {len(train_tasks)} / {len(val_tasks)} / {len(test_tasks)}")
print(f"MAX_LENGTH          : {MAX_LENGTH}")
print(f"Held-out test F1    : {overall_f1:.4f}")
print(f"Held-out test P / R : {overall_p:.4f} / {overall_r:.4f}")
print("=" * 60)


SUMMARY
Total tasks         : 24044
Train / Val / Test  : 19476 / 2164 / 2404
MAX_LENGTH          : 256
Held-out test F1    : 0.8718
Held-out test P / R : 0.8574 / 0.8867


In [25]:
from google.colab import files

files.download("/content/ner_output/final_model/model.safetensors")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
# ============================================================
# INFERENCE — predict entities on a new sentence
# ============================================================
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

# Reload from the saved final model (works even in a fresh session,
# as long as final_dir/ exists — no need to retrain)
final_dir = os.path.join(OUTPUT_DIR, "final_model")   # "./ner_output/final_model"
inf_tokenizer = AutoTokenizer.from_pretrained(final_dir)
inf_model = AutoModelForTokenClassification.from_pretrained(final_dir)
inf_model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
inf_model.to(device)


def predict_entities(text, model=inf_model, tokenizer=inf_tokenizer, max_length=MAX_LENGTH):
    enc = tokenizer(
        text,
        return_offsets_mapping=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )
    offsets = enc.pop("offset_mapping")[0].tolist()
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = model(**enc).logits[0]          # (seq_len, num_labels)
    pred_ids = torch.argmax(logits, dim=-1).tolist()
    id2label_local = model.config.id2label

    # Walk token predictions, merge consecutive B-/I- of the same tag into spans
    entities = []
    current = None  # {"label":..., "start":..., "end":...}

    for (start, end), pid in zip(offsets, pred_ids):
        if start == end:  # special tokens ([CLS], [SEP], padding)
            continue
        label = id2label_local[pid]

        if label == "O":
            if current:
                entities.append(current)
                current = None
            continue

        prefix, tag = label.split("-", 1)

        if prefix == "B" or current is None or current["label"] != tag:
            if current:
                entities.append(current)
            current = {"label": tag, "start": start, "end": end}
        else:  # "I-" continuing the same tag
            current["end"] = end

    if current:
        entities.append(current)

    # Attach the actual surface text for each span
    for ent in entities:
        ent["text"] = text[ent["start"]:ent["end"]]

    return entities


def show_predictions(text):
    ents = predict_entities(text)
    print(f"Text: {text}\n")
    if not ents:
        print("No entities found.")
        return
    print(f"{'Entity':35s} {'Tag':8s} {'Span'}")
    print("-" * 60)
    for e in ents:
        print(f"{e['text']:35s} {e['label']:8s} ({e['start']}, {e['end']})")


# ---------------- Demo ----------------
demo_sentence = "The recent judicial trajectory of Baksish Ahmad, a Border Security Force (BSF) constable dismissed from service in 2022 while posted in Narayanpur, Malda, serves as a primary case study. When Ahmad approached the Delhi High Court in 2025 to challenge his dismissal, the Division Bench dismissed his writ petition."
show_predictions(demo_sentence)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Text: The recent judicial trajectory of Baksish Ahmad, a Border Security Force (BSF) constable dismissed from service in 2022 while posted in Narayanpur, Malda, serves as a primary case study. When Ahmad approached the Delhi High Court in 2025 to challenge his dismissal, the Division Bench dismissed his writ petition.

Entity                              Tag      Span
------------------------------------------------------------
Baksish Ahmad                       PER      (34, 47)
Border Security Force               ORG      (51, 72)
BSF                                 ORG      (74, 77)
2022                                DATE     (115, 119)
Narayanpur                          LOC      (136, 146)
Malda                               LOC      (148, 153)
Ahmad                               PER      (192, 197)
Delhi High Court                    ORG      (213, 229)
2025                                DATE     (233, 237)
